# Notebook 1 — Exploratory Data Analysis
Price data for AAPL, MSFT, TSLA, NVDA, GOOGL (2018–2024)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
TICKERS = ['AAPL', 'MSFT', 'TSLA', 'NVDA', 'GOOGL']

## 1. Load price data

In [ ]:
dfs = {}
for t in TICKERS:
    path = Path(f'../raw_data/prices/{t}_2018_2024.csv')
    if path.exists():
        dfs[t] = pd.read_csv(path, parse_dates=['Date'])
        print(f'{t}: {len(dfs[t])} rows, {dfs[t]["Date"].min().date()} → {dfs[t]["Date"].max().date()}')
    else:
        print(f'[MISSING] {t} — run collect_price_data.py first')

## 2. Closing price over time

In [ ]:
fig, axes = plt.subplots(len(dfs), 1, figsize=(14, 3 * len(dfs)), sharex=True)
colors = ['#2563eb', '#16a34a', '#dc2626', '#7c3aed', '#ea580c']
for ax, (t, df), c in zip(axes, dfs.items(), colors):
    ax.plot(df['Date'], df['Close'], color=c, linewidth=1)
    ax.set_ylabel(f'{t} ($)', fontsize=10)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.suptitle('Adjusted Closing Prices 2018–2024', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Log returns distribution

In [ ]:
fig, axes = plt.subplots(1, len(dfs), figsize=(16, 4))
for ax, (t, df) in zip(axes, dfs.items()):
    log_ret = np.log(df['Close'] / df['Close'].shift(1)).dropna()
    ax.hist(log_ret, bins=80, edgecolor='none', color='#2563eb', alpha=0.7)
    ax.axvline(log_ret.mean(), color='red', linestyle='--', linewidth=1, label='Mean')
    ax.set_title(f'{t}\nμ={log_ret.mean():.4f} σ={log_ret.std():.4f}', fontsize=9)
    ax.set_xlabel('Log Return')
plt.suptitle('Daily Log Return Distributions', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Correlation matrix

In [ ]:
# Align all tickers on the same dates
closes = pd.DataFrame({t: df.set_index('Date')['Close'] for t, df in dfs.items()})
corr = closes.pct_change().corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Pairwise Return Correlation', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Technical indicator preview (AAPL)

In [ ]:
import sys
sys.path.insert(0, '../src/preprocessing')
from engineer_features import engineer_features, compute_rsi, compute_macd, compute_bollinger_bands

feat_path = Path('../processed_data/AAPL_features.csv')
if feat_path.exists():
    df_feat = pd.read_csv(feat_path, parse_dates=['Date'])
else:
    print('Run engineer_features.py first')
    df_feat = None

if df_feat is not None:
    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    recent = df_feat.tail(252)  # ~1 year
    axes[0].plot(recent['Date'], recent['Close'], '#2563eb'); axes[0].set_ylabel('Close')
    axes[1].plot(recent['Date'], recent['rsi_14'], '#dc2626'); axes[1].axhline(70, ls='--', c='gray'); axes[1].axhline(30, ls='--', c='gray'); axes[1].set_ylabel('RSI-14')
    axes[2].plot(recent['Date'], recent['macd'], '#16a34a', label='MACD'); axes[2].plot(recent['Date'], recent['macd_signal'], '#f59e0b', label='Signal'); axes[2].legend(fontsize=8); axes[2].set_ylabel('MACD')
    axes[3].plot(recent['Date'], recent['Close'], '#2563eb'); axes[3].plot(recent['Date'], recent['bb_upper'], '#94a3b8', ls='--'); axes[3].plot(recent['Date'], recent['bb_lower'], '#94a3b8', ls='--'); axes[3].set_ylabel('BB Bands')
    plt.suptitle('AAPL Technical Indicators (last 252 trading days)', fontsize=12)
    plt.tight_layout()
    plt.show()